#Importing libraries

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import pickle

from google.colab import drive
drive.mount("/content/gdrive", force_remount=True)

data_dir_path='/content/gdrive/My Drive/NYP ML PROJECT/Data/'

df = pd.read_csv(data_dir_path+'data_prepped.csv')


Mounted at /content/gdrive


In [ ]:
!pip install mlflow==2.5.0

!databricks configure --host https://community.cloud.databricks.com/


Username: loiuswang88@gmail.com
Password: 
Repeat for confirmation: 


In [ ]:

import mlflow
import mlflow.sklearn
mlflow.sklearn.autolog()

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Users/loiuswang88@gmail.com/4.1_Models")
#pw: nypSpotify!23

# mlflow.sklearn.autolog() #had to comment this bcos it was causing loops

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2629984321281848', creation_time=1692582629831, experiment_id='2629984321281848', last_update_time=1692630426992, lifecycle_stage='active', name='/Users/loiuswang88@gmail.com/4.1_Models', tags={'mlflow.experiment.sourceName': '/Users/loiuswang88@gmail.com/4.1_Models',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'loiuswang88@gmail.com',
 'mlflow.ownerId': '4845539840753118'}>

In [ ]:
#models which require scaling
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

#models which don't require scaling
from sklearn.ensemble import RandomForestClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import cross_val_score


#Data processing

In [ ]:
#encode key, mode and time signature
df = pd.get_dummies(df, columns=['key','mode','time_signature'])
df.head()

,year,track_name,artist_name,track_id,danceability,energy,loudness,speechiness,acousticness,instrumentalness,...,key_9,key_10,key_11,mode_0,mode_1,time_signature_0,time_signature_1,time_signature_3,time_signature_4,time_signature_5
0,2013,Mirrors,Justin Timberlake,4rHZZAmHpZrA3iH5zx8frV,0.574,0.512,-6.664,0.0503,0.23400,0.000000,...,0,0,0,1,0,0,0,0,1,0
1,2013,Blurred Lines,Robin Thicke,5PUvinSo4MNqW7vmomGRS7,0.861,0.504,-7.707,0.0489,0.00412,0.000018,...,0,0,0,0,1,0,0,0,1,0
2,2013,Radioactive,Imagine Dragons,4G8gkOterJn0Ywt6uhqbhp,0.448,0.784,-3.686,0.0627,0.10600,0.000108,...,1,0,0,0,1,0,0,0,1,0
3,2013,Wake Me Up,Avicii,0nrRP2bk19rLc0orkWPQk2,0.532,0.783,-5.697,0.0523,0.00380,0.001200,...,0,0,0,0,1,0,0,0,1,0
4,2013,Another Love,Tom Odell,3JvKfv6T31zO0ini8iNItO,0.445,0.537,-8.532,0.0400,0.69500,0.000017,...,0,0,0,1,0,0,0,0,1,0


In [ ]:
#conversion of loudness
def dB_to_linear(dB_value):
    return np.power(10, dB_value / 10)

df['linear_loudness'] = df['loudness'].apply(dB_to_linear)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5954 entries, 0 to 5953
Data columns (total 35 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              5954 non-null   int64  
 1   track_name        5954 non-null   object 
 2   artist_name       5954 non-null   object 
 3   track_id          5954 non-null   object 
 4   danceability      5954 non-null   float64
 5   energy            5954 non-null   float64
 6   loudness          5954 non-null   float64
 7   speechiness       5954 non-null   float64
 8   acousticness      5954 non-null   float64
 9   instrumentalness  5954 non-null   float64
 10  liveness          5954 non-null   float64
 11  valence           5954 non-null   float64
 12  tempo             5954 non-null   float64
 13  duration_ms       5954 non-null   int64  
 14  hit               5954 non-null   int64  
 15  key_0             5954 non-null   uint8  
 16  key_1             5954 non-null   uint8  


In [ ]:
df_target = df['hit']
df_feat = df.drop(['year','track_name','artist_name','track_id','loudness','hit'], axis=1)


#Feature Selection

In [ ]:
rf1 = RandomForestClassifier(n_estimators = 100, random_state=42)
rf1.fit(df_feat,df_target)

feature_importances = rf1.feature_importances_
sorted_indices = np.argsort(feature_importances)[::-1]

feature_names = df_feat.columns[:]

for i in sorted_indices:
    print(f"{feature_names[i]}: {feature_importances[i]}")

2023/08/21 15:18:35 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '522bbc81f1c44dc58304d0a48bafb0a2', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2023/08/21 15:18:35 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values 

danceability: 0.11105400530140766
instrumentalness: 0.1072955392652434
linear_loudness: 0.10526862333991975
energy: 0.10520315161850949
acousticness: 0.09292691461564086
duration_ms: 0.09107797746930743
valence: 0.07624057803224928
speechiness: 0.07580250322899583
tempo: 0.07155631988313896
liveness: 0.06829565395046916
mode_1: 0.008887069742160057
mode_0: 0.0083950607863005
key_1: 0.008377824831596773
key_0: 0.006898362098788783
key_8: 0.0064888287768556565
key_11: 0.00610999872102147
key_5: 0.005793574174726967
key_7: 0.005735209710963302
key_6: 0.005727755689066772
key_2: 0.005603583203575295
key_4: 0.005444422288443971
key_10: 0.005212553414747213
key_9: 0.005154802364214412
time_signature_4: 0.004280381123012063
key_3: 0.0033540608181441382
time_signature_3: 0.0023805646983660147
time_signature_5: 0.0011474857687365277
time_signature_1: 0.0002714305648285876
time_signature_0: 1.576451956966506e-05


In [ ]:
#stratified sampling
from sklearn.model_selection import StratifiedShuffleSplit
#initialization a generator
strat_shuff_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Get the index values from the generator
train_idx, test_idx = next(strat_shuff_split.split(df_feat, df_target))
train_idx, test_idx

X_train = df_feat.loc[train_idx]
Y_train = df_target.loc[train_idx]

X_test = df_feat.loc[test_idx]
Y_test = df_target.loc[test_idx]



In [ ]:
rf1 = RandomForestClassifier(n_estimators = 100, random_state=42)
rf1.fit(X_train,Y_train)

feature_importances = rf1.feature_importances_
sorted_indices = np.argsort(feature_importances)[::-1]

feature_names = df_feat.columns[:]

for i in sorted_indices:
    print(f"{feature_names[i]}: {feature_importances[i]}")

2023/08/21 15:18:52 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '3912576802104f0ca408eab29dbc5eb1', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2023/08/21 15:18:52 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values 

linear_loudness: 0.1092166552998602
danceability: 0.1090242768325729
energy: 0.10400047797028635
instrumentalness: 0.10351847864217795
acousticness: 0.09289044322851052
duration_ms: 0.08807045182930213
valence: 0.0813209185230726
speechiness: 0.07486895275779652
tempo: 0.07159597348068443
liveness: 0.06889954987663321
mode_1: 0.008438347478112715
mode_0: 0.00833993596421045
key_1: 0.007954824213359224
key_0: 0.00729342476806248
key_8: 0.006890647979360604
key_2: 0.00633039065100863
key_9: 0.006155150211532668
key_5: 0.006035577691047133
key_11: 0.005979551675576038
key_6: 0.005786933436542832
time_signature_4: 0.005411065104442617
key_7: 0.0053920245118058594
key_4: 0.005019789418769146
key_10: 0.00469967708908603
key_3: 0.0029646168370304317
time_signature_3: 0.002577205636845985
time_signature_5: 0.001004307766629957
time_signature_1: 0.000288562634275512
time_signature_0: 3.1788491404827076e-05


In [ ]:
#we can then reduce the number of features for training.. to the first 10
set_A = ['danceability','energy','linear_loudness','speechiness','acousticness','instrumentalness','liveness','tempo','valence','duration_ms']
#set_B is just the first 4 features
set_B = ['linear_loudness','danceability','energy','instrumentalness']


#Training using Feature Set A


In [ ]:
#stratify split

#initialization a generator
strat_shuff_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Get the index values from the generator
train_idx, test_idx = next(strat_shuff_split.split(df[set_A], df['hit']))
train_idx, test_idx

X_train = df.loc[train_idx, set_A]
Y_train = df.loc[train_idx, 'hit']

X_test = df.loc[test_idx, set_A]
Y_test = df.loc[test_idx, 'hit']


In [ ]:
#Log regression and KNN
current_run = mlflow.start_run()

# Create a pipeline - Log regression
print("Log regression")
pipeline_lr = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', LogisticRegression())
])

# Cross-validation
cv_scores = cross_val_score(pipeline_lr, X_train, Y_train, cv=5)

# Train the pipeline on the entire training set
pipeline_lr.fit(X_train, Y_train)

mean_cv_score = np.mean(cv_scores)
std_cv_score = np.std(cv_scores)

print(mean_cv_score)
print(std_cv_score)

predictions = pipeline_lr.predict(X_test)
print('Logistic regression accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

#use model to predict probability that given y value is 1
y_pred_proba = pipeline_lr.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()

Log regression


2023/08/21 15:19:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:19:13 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/mod

0.840855282301798
0.003957340092314409


2023/08/21 15:19:17 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Logistic regression accuracy 0.837951301427372
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.98      0.91      1000
           1       0.48      0.12      0.19       191

    accuracy                           0.84      1191
   macro avg       0.67      0.55      0.55      1191
weighted avg       0.79      0.84      0.79      1191

AUC:
0.8554764397905759


In [ ]:
# Get predicted probabilities
predicted_probabilities = pipeline_lr.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics for different thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
for threshold in thresholds:
    predictions = (predicted_probabilities > threshold).astype(int)
    accuracy = accuracy_score(Y_test, predictions)
    precision = precision_score(Y_test, predictions)
    recall = recall_score(Y_test, predictions)
    f1 = f1_score(Y_test, predictions)
    print(f"Threshold: {threshold:.2f} | Accuracy: {accuracy:.2f} | Precision: {precision:.2f} | Recall: {recall:.2f} | F1-Score: {f1:.2f}")

2023/08/21 15:19:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Threshold: 0.30 | Accuracy: 0.83 | Precision: 0.47 | Recall: 0.51 | F1-Score: 0.49
Threshold: 0.40 | Accuracy: 0.85 | Precision: 0.54 | Recall: 0.28 | F1-Score: 0.37
Threshold: 0.50 | Accuracy: 0.84 | Precision: 0.48 | Recall: 0.12 | F1-Score: 0.19
Threshold: 0.60 | Accuracy: 0.84 | Precision: 0.43 | Recall: 0.03 | F1-Score: 0.06
Threshold: 0.70 | Accuracy: 0.84 | Precision: 0.50 | Recall: 0.01 | F1-Score: 0.02


In [ ]:
current_run = mlflow.start_run()


# Create a pipeline - KNN
print("\n KNN")
pipeline_knn = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', KNeighborsClassifier())
])

# Cross-validation
cv_scores = cross_val_score(pipeline_knn, X_train, Y_train, cv=5)

# Train the pipeline on the entire training set
pipeline_knn.fit(X_train, Y_train)

mean_cv_score = np.mean(cv_scores)
std_cv_score = np.std(cv_scores)

print(mean_cv_score)
print(std_cv_score)

predictions = pipeline_knn.predict(X_test)
print('KNN accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

#use model to predict probability that given y value is 1
y_pred_proba = pipeline_knn.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()



 KNN


2023/08/21 15:19:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:19:24 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/mod

0.8276314513213471
0.004832042069078829


2023/08/21 15:19:28 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


KNN accuracy 0.8396305625524769
Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      1000
           1       0.50      0.34      0.40       191

    accuracy                           0.84      1191
   macro avg       0.69      0.64      0.65      1191
weighted avg       0.82      0.84      0.83      1191

AUC:
0.8003664921465969


In [ ]:
current_run = mlflow.start_run()

# Create a pipeline
pipeline_svc = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', SVC(kernel='rbf', C=0.01, gamma=0.01,random_state=42))
])

# Cross-validation
cv_scores = cross_val_score(pipeline_svc, X_train, Y_train, cv=5)

# Train the pipeline on the entire training set
pipeline_svc.fit(X_train, Y_train)

mean_cv_score = np.mean(cv_scores)
std_cv_score = np.std(cv_scores)

print(mean_cv_score)
print(std_cv_score)

predictions = pipeline_svc.predict(X_test)
print('SVM accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

#y_pred_proba = pipeline.predict_proba(X_test)[::,1]
#auc = roc_auc_score(Y_test, y_pred_proba)
#print("AUC:")
#print(auc)


mlflow.end_run()

2023/08/21 15:19:30 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in

0.8398070665831915
0.0004319821074154611
SVM accuracy 0.8396305625524769
Classification Report:
              precision    recall  f1-score   support

           0       0.84      1.00      0.91      1000
           1       0.00      0.00      0.00       191

    accuracy                           0.84      1191
   macro avg       0.42      0.50      0.46      1191
weighted avg       0.70      0.84      0.77      1191



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
#trying RF, decision tree, adaboost, XGboost using the un-SMOTE datasets

current_run = mlflow.start_run()
print("\n Random Forest")
# Initialize the Random Forest classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Perform cross-validation with 5 folds (you can adjust the number of folds)
cv_scores = cross_val_score(rf, X_train, Y_train, cv=5, scoring='accuracy')
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

rf.fit(X_train,Y_train)

predictions = rf.predict(X_test)
print('Random Forest accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = rf.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()


 Random Forest


2023/08/21 15:19:47 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Cross-validation scores: [0.87513116 0.86149003 0.85834208 0.85084034 0.86554622]
Mean CV score: 0.8622699656987663


2023/08/21 15:19:52 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/models/signature.py:152: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:19:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/p

Random Forest accuracy 0.8673383711167086
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.97      0.92      1000
           1       0.67      0.35      0.46       191

    accuracy                           0.87      1191
   macro avg       0.78      0.66      0.69      1191
weighted avg       0.85      0.87      0.85      1191

AUC:
0.8947722513089006


In [ ]:
current_run = mlflow.start_run()
#Decision Tree
print("\n Decision Tree")
dt = DecisionTreeClassifier(random_state=42)

cv_scores = cross_val_score(dt, X_train, Y_train, cv=5, scoring='accuracy')
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

dt.fit(X_train,Y_train)

predictions = dt.predict(X_test)
print('Decision Tree accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = dt.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()


 Decision Tree


2023/08/21 15:19:58 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Cross-validation scores: [0.79643232 0.80272823 0.80482686 0.80147059 0.79831933]
Mean CV score: 0.8007554648302133


2023/08/21 15:19:59 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/models/signature.py:152: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:20:03 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/p

Decision Tree accuracy 0.8178001679261125
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.89      0.89      1000
           1       0.43      0.45      0.44       191

    accuracy                           0.82      1191
   macro avg       0.66      0.67      0.67      1191
weighted avg       0.82      0.82      0.82      1191

AUC:
0.669130890052356


In [ ]:
current_run = mlflow.start_run()
#Ada Boost
print("\n Ada Boost")
# Initialize the AdaBoost classifier with DecisionTreeClassifier as the base estimator
base_estimator = DecisionTreeClassifier(max_depth=1)
ada = AdaBoostClassifier(estimator=base_estimator, n_estimators=100, random_state=42)

# Perform cross-validation with 5 folds (you can adjust the number of folds)
cv_scores = cross_val_score(ada, X_train, Y_train, cv=5, scoring='accuracy')
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

ada.fit(X_train,Y_train)

predictions = ada.predict(X_test)
print('Ada Boost accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = ada.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()


 Ada Boost


2023/08/21 15:20:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Cross-validation scores: [0.86358867 0.8478489  0.85309549 0.85189076 0.86134454]
Mean CV score: 0.8555536695265726


2023/08/21 15:20:14 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/models/signature.py:152: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:20:17 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/p

Ada Boost accuracy 0.8639798488664987
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      1000
           1       0.60      0.45      0.51       191

    accuracy                           0.86      1191
   macro avg       0.75      0.69      0.72      1191
weighted avg       0.85      0.86      0.86      1191

AUC:
0.8743821989528796


In [ ]:
current_run = mlflow.start_run()
#XG Boost
print("\n XG Boost")
xgb = XGBClassifier()
cv_scores = cross_val_score(xgb, X_train, Y_train, cv=5, scoring='accuracy')

# Print the cross-validation scores
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

xgb.fit(X_train, Y_train)

predictions = xgb.predict(X_test)
print('XG Boost accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = ada.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()


 XG Boost
Cross-validation scores: [0.86253935 0.84679958 0.85309549 0.85819328 0.8539916 ]
Mean CV score: 0.8549238583156242


2023/08/21 15:20:27 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


XG Boost accuracy 0.8706968933669186
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.95      0.93      1000
           1       0.64      0.44      0.52       191

    accuracy                           0.87      1191
   macro avg       0.77      0.70      0.72      1191
weighted avg       0.86      0.87      0.86      1191

AUC:
0.8743821989528796


In [ ]:
# this cell is the same code as the cell above, but added mlflow logging and putting the 4 models in a for loop
# import mlflow
# import mlflow.sklearn
# from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.model_selection import cross_val_score
# from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
# from xgboost import XGBClassifier

# # Start an MLflow run
# with mlflow.start_run():
#     # Enable autologging for scikit-learn models
#     mlflow.sklearn.autolog()

#     # Random Forest
#     print("\n Random Forest")
#     rf = RandomForestClassifier(n_estimators=100, random_state=42)
#     cv_scores_rf = cross_val_score(rf, X_train, Y_train, cv=5, scoring='accuracy')
#     print("Cross-validation scores:", cv_scores_rf)
#     print("Mean CV score:", cv_scores_rf.mean())
#     rf.fit(X_train, Y_train)
#     predictions_rf = rf.predict(X_test)
#     print('Random Forest accuracy', accuracy_score(Y_test, predictions_rf))
#     print("Classification Report:")
#     print(classification_report(Y_test, predictions_rf))
#     y_pred_proba_rf = rf.predict_proba(X_test)[::, 1]
#     auc_rf = roc_auc_score(Y_test, y_pred_proba_rf)
#     print("AUC:", auc_rf)

#     # Decision Tree
#     print("\n Decision Tree")
#     dt = DecisionTreeClassifier(random_state=42)
#     cv_scores_dt = cross_val_score(dt, X_train, Y_train, cv=5, scoring='accuracy')
#     print("Cross-validation scores:", cv_scores_dt)
#     print("Mean CV score:", cv_scores_dt.mean())
#     dt.fit(X_train, Y_train)
#     predictions_dt = dt.predict(X_test)
#     print('Decision Tree accuracy', accuracy_score(Y_test, predictions_dt))
#     print("Classification Report:")
#     print(classification_report(Y_test, predictions_dt))
#     y_pred_proba_dt = dt.predict_proba(X_test)[::, 1]
#     auc_dt = roc_auc_score(Y_test, y_pred_proba_dt)
#     print("AUC:", auc_dt)

#     # Ada Boost
#     print("\n Ada Boost")
#     base_estimator = DecisionTreeClassifier(max_depth=1)
#     ada = AdaBoostClassifier(estimator=base_estimator, n_estimators=50, random_state=42)
#     cv_scores_ada = cross_val_score(ada, X_train, Y_train, cv=5, scoring='accuracy')
#     print("Cross-validation scores:", cv_scores_ada)
#     print("Mean CV score:", cv_scores_ada.mean())
#     ada.fit(X_train, Y_train)
#     predictions_ada = ada.predict(X_test)
#     print('Ada Boost accuracy', accuracy_score(Y_test, predictions_ada))
#     print("Classification Report:")
#     print(classification_report(Y_test, predictions_ada))
#     y_pred_proba_ada = ada.predict_proba(X_test)[::, 1]
#     auc_ada = roc_auc_score(Y_test, y_pred_proba_ada)
#     print("AUC:", auc_ada)

#     # XG Boost
#     print("\n XG Boost")
#     xgb_model = XGBClassifier()
#     cv_scores_xgb = cross_val_score(xgb_model, X_train, Y_train, cv=5, scoring='accuracy')
#     print("Cross-validation scores:", cv_scores_xgb)
#     print("Mean CV score:", cv_scores_xgb.mean())
#     xgb_model.fit(X_train, Y_train)
#     predictions_xgb = xgb_model.predict(X_test)
#     print('XG Boost accuracy', accuracy_score(Y_test, predictions_xgb))
#     print("Classification Report:")
#     print(classification_report(Y_test, predictions_xgb))
#     y_pred_proba_xgb = xgb_model.predict_proba(X_test)[::, 1]
#     auc_xgb = roc_auc_score(Y_test, y_pred_proba_xgb)
#     print("AUC:", auc_xgb)


In [ ]:
# #balanced random forest
# current_run = mlflow.start_run()

# import warnings
# warnings.filterwarnings("ignore")

# print("\n Balanced Random Forest")
# # Initialize the Random Forest classifier
# brf = BalancedRandomForestClassifier(n_estimators=100, sampling_strategy='not majority', random_state=42)

# # Perform cross-validation with 5 folds (you can adjust the number of folds)
# cv_scores = cross_val_score(brf, X_train, Y_train, cv=5, scoring='accuracy')
# print("Cross-validation scores:", cv_scores)
# print("Mean CV score:", cv_scores.mean())

# brf.fit(X_train,Y_train)

# predictions = brf.predict(X_test)
# print('Random Forest accuracy', accuracy_score(Y_test, predictions))

# print("Classification Report:")
# print(classification_report(Y_test, predictions))

# y_pred_proba = brf.predict_proba(X_test)[::,1]
# auc = roc_auc_score(Y_test, y_pred_proba)
# print("AUC:")
# print(auc)

# mlflow.end_run()

#Oversampling

In [ ]:
Y_train.value_counts()

0    4000
1     763
Name: hit, dtype: int64

In [ ]:
#now we try to use SMOTE to create a more balanced dataset
#best performing models so far: RF, XG, ADA
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy='auto',random_state=42)

X_train_resampled, Y_train_resampled = smote.fit_resample(X_train, Y_train)

2023/08/21 15:20:27 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'f01fdae6273e42b9a9ec12ef9c4645db', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2023/08/21 15:20:28 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.
2023/08/21 15:20:28 WARNING mlflow.sklearn: Failed to infer model signature: the trained model does not specify a `predict` function, which is required in order to infer the signature
2023/08/21 15:20:28 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


In [ ]:
Y_train_resampled.value_counts()

0    4000
1    4000
Name: hit, dtype: int64

In [ ]:
# Initialize the Random Forest classifier
current_run = mlflow.start_run()

rf_smote = RandomForestClassifier(n_estimators=200, random_state=42)

# Perform cross-validation with 5 folds (you can adjust the number of folds)
cv_scores = cross_val_score(rf_smote, X_train_resampled, Y_train_resampled, cv=5, scoring='accuracy')
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

rf_smote.fit(X_train_resampled,Y_train_resampled)

predictions = rf_smote.predict(X_test)
print('Random Forest accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = rf_smote.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()

2023/08/21 15:21:01 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Cross-validation scores: [0.835625 0.92125  0.919375 0.921875 0.913125]
Mean CV score: 0.9022500000000001


2023/08/21 15:21:13 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/models/signature.py:152: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:21:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/p

Random Forest accuracy 0.8446683459277917
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.89      0.91      1000
           1       0.51      0.62      0.56       191

    accuracy                           0.84      1191
   macro avg       0.72      0.75      0.73      1191
weighted avg       0.86      0.84      0.85      1191

AUC:
0.8887539267015707


In [ ]:
#using random oversampler
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(sampling_strategy="not majority")
X_res, Y_res = ros.fit_resample(X_train, Y_train)

In [ ]:
current_run = mlflow.start_run()

# Initialize the Random Forest classifier
rf_ros = RandomForestClassifier(n_estimators=100, random_state=42)

# Perform cross-validation with 5 folds (you can adjust the number of folds)
cv_scores = cross_val_score(rf_ros, X_res, Y_res, cv=5, scoring='accuracy')
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

rf_ros.fit(X_res.values, Y_res.values)

predictions = rf_ros.predict(X_test)
print('Random Forest accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = rf_ros.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

#dumping the model
#pickle.dump(rf_ros, open('model.pkl', 'wb'))

mlflow.end_run()

Cross-validation scores: [0.94625  0.95375  0.95875  0.958125 0.95375 ]
Mean CV score: 0.954125


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
2023/08/21 15:21:41 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ 

Random Forest accuracy 0.8723761544920235
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.94      0.93      1000
           1       0.63      0.51      0.56       191

    accuracy                           0.87      1191
   macro avg       0.77      0.72      0.74      1191
weighted avg       0.86      0.87      0.87      1191

AUC:
0.8970026178010472


-> random over sampling with random forest - good f1 scores

In [ ]:
#tuning the hyperparameters in randomforest

# grid_space={'max_depth':[3,5,10],
#               'max_features':[5,7,9],
#               'min_samples_leaf':[1,2,3],
#               'min_samples_split':[1,2,3]
#            }

# from sklearn.model_selection import GridSearchCV

# rf_gs = RandomForestClassifier()

# grid = GridSearchCV(rf_gs,param_grid=grid_space,cv=5,scoring='accuracy')
# model_grid = grid.fit(X_res, y_res)

# print('Best hyperparameters are: '+str(model_grid.best_params_))
# print('Best score is: '+str(model_grid.best_score_))

# Best hyperparameters are: {'max_depth': 10, 'max_features': 7, 'min_samples_leaf': 1, 'min_samples_split': 3}
# Best score is: 0.9088750000000001

In [ ]:
#using the tuned hyperparameters
current_run = mlflow.start_run()

rf_opt = RandomForestClassifier(n_estimators=100, max_depth=10, max_features=7, min_samples_leaf=1, min_samples_split=3, random_state=42)

# Perform cross-validation with 5 folds (you can adjust the number of folds)
cv_scores = cross_val_score(rf_opt, X_res, Y_res, cv=5, scoring='accuracy')
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

rf_opt.fit(X_res, Y_res)

predictions = rf_opt.predict(X_test)
print('Random Forest accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = rf_opt.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)


mlflow.end_run()


2023/08/21 15:21:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Cross-validation scores: [0.903125 0.91125  0.91375  0.894375 0.90125 ]
Mean CV score: 0.9047499999999999


2023/08/21 15:22:03 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/models/signature.py:152: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:22:07 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/p

Random Forest accuracy 0.8320738874895046
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.84      0.89      1000
           1       0.49      0.78      0.60       191

    accuracy                           0.83      1191
   macro avg       0.72      0.81      0.75      1191
weighted avg       0.88      0.83      0.85      1191

AUC:
0.8922748691099477


Good CV scores but not as good during testing.
Recall is increased but precision drops further

In [ ]:
current_run = mlflow.start_run()

#Ada Boost
# Initialize the AdaBoost classifier with DecisionTreeClassifier as the base estimator
base_estimator = DecisionTreeClassifier(max_depth=1)
ada_smote = AdaBoostClassifier(estimator=base_estimator, n_estimators=100, random_state=42)

# Perform cross-validation with 5 folds (you can adjust the number of folds)
cv_scores = cross_val_score(ada_smote, X_train_resampled, Y_train_resampled, cv=5, scoring='accuracy')
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

ada_smote.fit(X_train_resampled,Y_train_resampled)

predictions = ada_smote.predict(X_test)
print('Ada Boost accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = ada_smote.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)


mlflow.end_run()

2023/08/21 15:22:16 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Cross-validation scores: [0.751875 0.873125 0.873125 0.8775   0.86625 ]
Mean CV score: 0.8483750000000001


2023/08/21 15:22:21 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/models/signature.py:152: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:22:25 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/p

Ada Boost accuracy 0.8144416456759026
Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.84      0.88      1000
           1       0.45      0.68      0.54       191

    accuracy                           0.81      1191
   macro avg       0.69      0.76      0.71      1191
weighted avg       0.85      0.81      0.83      1191

AUC:
0.8672434554973822


In [ ]:
#X_res, Y_res
current_run = mlflow.start_run()

#Ada Boost
# Initialize the AdaBoost classifier with DecisionTreeClassifier as the base estimator
base_estimator = DecisionTreeClassifier(max_depth=1)
ada_ros = AdaBoostClassifier(estimator=base_estimator, n_estimators=100, random_state=42)

# Perform cross-validation with 5 folds (you can adjust the number of folds)
cv_scores = cross_val_score(ada_ros, X_res, Y_res, cv=5, scoring='accuracy')
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

ada_ros.fit(X_res, Y_res)

predictions = ada_ros.predict(X_test)
print('Ada Boost accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = ada_ros.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)


mlflow.end_run()

2023/08/21 15:22:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Cross-validation scores: [0.82     0.8275   0.840625 0.83875  0.829375]
Mean CV score: 0.83125


2023/08/21 15:22:39 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/models/signature.py:152: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:22:42 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/p

Ada Boost accuracy 0.8052057094878253
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.81      0.87      1000
           1       0.44      0.78      0.56       191

    accuracy                           0.81      1191
   macro avg       0.70      0.80      0.72      1191
weighted avg       0.87      0.81      0.82      1191

AUC:
0.8763717277486911


In [ ]:
current_run = mlflow.start_run()

#XG Boost
xgb_smote = XGBClassifier()
cv_scores = cross_val_score(xgb_smote, X_train_resampled, Y_train_resampled, cv=5, scoring='accuracy')

# Print the cross-validation scores
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

xgb_smote.fit(X_train_resampled, Y_train_resampled)

predictions = xgb_smote.predict(X_test)
print('XG Boost boost accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = xgb_smote.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)


mlflow.end_run()

Cross-validation scores: [0.81625  0.919375 0.926875 0.911875 0.91375 ]
Mean CV score: 0.897625
XG Boost boost accuracy 0.8404701931150294
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.89      0.90      1000
           1       0.50      0.60      0.55       191

    accuracy                           0.84      1191
   macro avg       0.71      0.74      0.72      1191
weighted avg       0.85      0.84      0.85      1191

AUC:
0.8815287958115184


In [ ]:
#X_res, Y_res
current_run = mlflow.start_run()

#XG Boost
xgb_ros = XGBClassifier()
cv_scores = cross_val_score(xgb_ros, X_res, Y_res, cv=5, scoring='accuracy')

# Print the cross-validation scores
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

xgb_ros.fit(X_res, Y_res)

predictions = xgb_ros.predict(X_test)
print('XG Boost boost accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = xgb_ros.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)


mlflow.end_run()

Cross-validation scores: [0.940625 0.944375 0.95     0.93625  0.939375]
Mean CV score: 0.9421250000000001
XG Boost boost accuracy 0.8530646515533166
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.91      0.91      1000
           1       0.54      0.58      0.56       191

    accuracy                           0.85      1191
   macro avg       0.73      0.74      0.73      1191
weighted avg       0.86      0.85      0.86      1191

AUC:
0.8847015706806283


In [ ]:
current_run = mlflow.start_run()

#logistic regression with scaler
# Create a pipeline
pipeline_lr_smote = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', LogisticRegression())
])

# Cross-validation
cv_scores = cross_val_score(pipeline_lr_smote, X_train_resampled, Y_train_resampled, cv=5)

# Train the pipeline on the entire training set
pipeline_lr_smote.fit(X_train_resampled, Y_train_resampled)

mean_cv_score = np.mean(cv_scores)
std_cv_score = np.std(cv_scores)

print(mean_cv_score)
print(std_cv_score)

predictions = pipeline_lr_smote.predict(X_test)
print('Logistic regression accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = pipeline_lr_smote.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()

2023/08/21 15:23:02 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:23:06 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/mod

0.7842499999999999
0.012633041993122627


2023/08/21 15:23:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Logistic regression accuracy 0.7405541561712846
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.72      0.82      1000
           1       0.37      0.84      0.51       191

    accuracy                           0.74      1191
   macro avg       0.66      0.78      0.67      1191
weighted avg       0.86      0.74      0.77      1191

AUC:
0.8543350785340315


In [ ]:
#X_res, Y_res
current_run = mlflow.start_run()

#logistic regression with scaler
# Create a pipeline
pipeline_lr_ros = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', LogisticRegression())
])

# Cross-validation
cv_scores = cross_val_score(pipeline_lr_ros, X_res, Y_res, cv=5)

# Train the pipeline on the entire training set
pipeline_lr_ros.fit(X_res, Y_res)

mean_cv_score = np.mean(cv_scores)
std_cv_score = np.std(cv_scores)

print(mean_cv_score)
print(std_cv_score)

predictions = pipeline_lr_ros.predict(X_test)
print('Logistic regression accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = pipeline_lr_ros.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()

2023/08/21 15:23:11 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:23:13 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/mod

0.75975
0.0069439722061655825


2023/08/21 15:23:17 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Logistic regression accuracy 0.7296389588581025
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.71      0.81      1000
           1       0.36      0.85      0.50       191

    accuracy                           0.73      1191
   macro avg       0.66      0.78      0.66      1191
weighted avg       0.86      0.73      0.76      1191

AUC:
0.8554869109947644


In [ ]:
current_run = mlflow.start_run()

#KNN
# Create a pipeline
pipeline_knn_smote = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', KNeighborsClassifier())
])

# Cross-validation
cv_scores = cross_val_score(pipeline_knn_smote, X_train_resampled, Y_train_resampled, cv=5)

# Train the pipeline on the entire training set
pipeline_knn_smote.fit(X_train_resampled, Y_train_resampled)

mean_cv_score = np.mean(cv_scores)
std_cv_score = np.std(cv_scores)

print(mean_cv_score)
print(std_cv_score)

predictions = pipeline_knn_smote.predict(X_test)
print('KNN accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = pipeline_knn_smote.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()

2023/08/21 15:23:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:23:26 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/mod

0.8306250000000001
0.013422229695546134


2023/08/21 15:23:31 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


KNN accuracy 0.7497900923593619
Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.75      0.83      1000
           1       0.37      0.77      0.50       191

    accuracy                           0.75      1191
   macro avg       0.66      0.76      0.67      1191
weighted avg       0.85      0.75      0.78      1191

AUC:
0.818633507853403


In [ ]:
#X_res, Y_res
current_run = mlflow.start_run()

#KNN
# Create a pipeline
pipeline_knn_ros = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', KNeighborsClassifier())
])

# Cross-validation
cv_scores = cross_val_score(pipeline_knn_ros, X_res, Y_res, cv=5)

# Train the pipeline on the entire training set
pipeline_knn_ros.fit(X_res, Y_res)

mean_cv_score = np.mean(cv_scores)
std_cv_score = np.std(cv_scores)

print(mean_cv_score)
print(std_cv_score)

predictions = pipeline_knn_ros.predict(X_test)
print('KNN accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = pipeline_knn_ros.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()

2023/08/21 15:23:32 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:23:39 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/mod

0.8292499999999998
0.009853616087508182


2023/08/21 15:23:43 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


KNN accuracy 0.7338371116708649
Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.74      0.82      1000
           1       0.34      0.72      0.47       191

    accuracy                           0.73      1191
   macro avg       0.64      0.73      0.64      1191
weighted avg       0.84      0.73      0.77      1191

AUC:
0.7788246073298429


#Undersampling

In [ ]:
#since oversampling gave worse results, we try undersampling instead
from imblearn.under_sampling import RandomUnderSampler

# Initialize the RandomUnderSampler
rus = RandomUnderSampler(sampling_strategy=1, random_state=42)

# Perform undersampling on the training data
X_train_undersampled, Y_train_undersampled = rus.fit_resample(X_train, Y_train)

Y_train_undersampled.value_counts()

0    763
1    763
Name: hit, dtype: int64

In [ ]:
current_run = mlflow.start_run()

# Initialize the Random Forest classifier
rf_rus = RandomForestClassifier(n_estimators=100, random_state=42)

# Perform cross-validation with 5 folds (you can adjust the number of folds)
cv_scores = cross_val_score(rf_rus, X_train_undersampled, Y_train_undersampled, cv=5, scoring='accuracy')
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

rf_rus.fit(X_train_undersampled,Y_train_undersampled)

predictions = rf_rus.predict(X_test)
print('Random Forest accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = rf_rus.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()

2023/08/21 15:23:46 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Cross-validation scores: [0.79738562 0.7442623  0.79016393 0.7704918  0.76393443]
Mean CV score: 0.7732476159862852


2023/08/21 15:23:50 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/models/signature.py:152: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:23:55 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/p

Random Forest accuracy 0.8110831234256927
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.80      0.88      1000
           1       0.45      0.85      0.59       191

    accuracy                           0.81      1191
   macro avg       0.71      0.83      0.73      1191
weighted avg       0.88      0.81      0.83      1191

AUC:
0.8956439790575916


In [ ]:
current_run = mlflow.start_run()

#Ada Boost
# Initialize the AdaBoost classifier with DecisionTreeClassifier as the base estimator
base_estimator = DecisionTreeClassifier(max_depth=1)
ada_rus = AdaBoostClassifier(estimator=base_estimator, n_estimators=100, random_state=42)

# Perform cross-validation with 5 folds (you can adjust the number of folds)
cv_scores = cross_val_score(ada_rus, X_train_undersampled, Y_train_undersampled, cv=5, scoring='accuracy')
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

ada_rus.fit(X_train_undersampled,Y_train_undersampled)

predictions = ada_rus.predict(X_test)
print('Ada Boost accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = ada_rus.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)


mlflow.end_run()

2023/08/21 15:23:58 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Cross-validation scores: [0.81699346 0.73114754 0.78688525 0.77704918 0.73770492]
Mean CV score: 0.7699560698596379


2023/08/21 15:24:02 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/models/signature.py:152: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2023/08/21 15:24:06 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/p

Ada Boost accuracy 0.7758186397984886
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.77      0.85      1000
           1       0.40      0.82      0.54       191

    accuracy                           0.78      1191
   macro avg       0.68      0.79      0.70      1191
weighted avg       0.87      0.78      0.80      1191

AUC:
0.8748089005235602


In [ ]:
current_run = mlflow.start_run()

#XG Boost
xgb_rus = XGBClassifier()
cv_scores = cross_val_score(xgb_rus, X_train_undersampled, Y_train_undersampled, cv=5, scoring='accuracy')

# Print the cross-validation scores
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

xgb_rus.fit(X_train_undersampled, Y_train_undersampled)

predictions = xgb_rus.predict(X_test)
print('XG Boost boost accuracy', accuracy_score(Y_test, predictions))

print("Classification Report:")
print(classification_report(Y_test, predictions))

y_pred_proba = xgb_rus.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()

Cross-validation scores: [0.80065359 0.74754098 0.80327869 0.78360656 0.7704918 ]
Mean CV score: 0.7811143255116255
XG Boost boost accuracy 0.7892527287993283
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.78      0.86      1000
           1       0.42      0.82      0.56       191

    accuracy                           0.79      1191
   macro avg       0.69      0.80      0.71      1191
weighted avg       0.87      0.79      0.81      1191

AUC:
0.8788534031413613


#Ensemble Models

In [ ]:
current_run = mlflow.start_run()

#trying stack classifier with 2 bases
base_clf1 = RandomForestClassifier(n_estimators=100, random_state=42)
base_clf2 = AdaBoostClassifier(estimator=base_estimator, n_estimators=100, random_state=42)
#base_clf3 = XGBClassifier()

from sklearn.ensemble import StackingClassifier

# Initialize the StackingClassifier
#estimators = [('rf', base_clf1), ('ada', base_clf2), ('xgb', base_clf3)]
estimators = [('rf', base_clf1), ('ada', base_clf2)]

#meta_clf = GradientBoostingClassifier(learning_rate=0.01, n_estimators=50)
meta_clf = XGBClassifier()

stacking_clf_2 = StackingClassifier(estimators=estimators, final_estimator=meta_clf)

stacking_clf_2.fit(X_train,Y_train)

predictions = stacking_clf_2.predict(X_test)
print(classification_report(Y_test, predictions))

y_pred_proba = stacking_clf_2.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()

2023/08/21 15:24:10 WARNING mlflow.utils: Truncated the value of the key `final_estimator`. Truncated value: `XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
             ...`
2023/08/21 15:24:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.10/dist-packages/mlflow/data/pandas_dataset.py:116: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floa

              precision    recall  f1-score   support

           0       0.89      0.94      0.91      1000
           1       0.54      0.37      0.44       191

    accuracy                           0.85      1191
   macro avg       0.71      0.66      0.68      1191
weighted avg       0.83      0.85      0.84      1191

AUC:
0.8692617801047121


In [ ]:
#stacking with 3 bases
current_run = mlflow.start_run()

#trying stack classifier
base_clf1 = RandomForestClassifier(n_estimators=100, random_state=42)
base_clf2 = AdaBoostClassifier(estimator=base_estimator, n_estimators=50, random_state=42)
base_clf3 = XGBClassifier()

from sklearn.ensemble import StackingClassifier

# Initialize the StackingClassifier
estimators = [('rf', base_clf1), ('ada', base_clf2), ('xgb', base_clf3)]
#estimators = [('rf', base_clf1), ('ada', base_clf2)]

meta_clf = GradientBoostingClassifier(learning_rate=0.01, n_estimators=50)
#meta_clf = XGBClassifier()

stacking_clf_3 = StackingClassifier(estimators=estimators, final_estimator=meta_clf)

stacking_clf_3.fit(X_train,Y_train)

predictions = stacking_clf_3.predict(X_test)
print(classification_report(Y_test, predictions))

y_pred_proba = stacking_clf_3.predict_proba(X_test)[::,1]
auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()

2023/08/21 15:24:33 WARNING mlflow.utils: Truncated the value of the key `estimators`. Truncated value: `[('rf', RandomForestClassifier(random_state=42)), ('ada', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                   random_state=42)), ('xgb', XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_id=None, grow_policy=None, impor...`
2023/08/21 15:24:33 WARNING mlflow.utils: Truncated the value of the key `xgb`. Truncated value: `XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_

              precision    recall  f1-score   support

           0       0.84      1.00      0.91      1000
           1       0.00      0.00      0.00       191

    accuracy                           0.84      1191
   macro avg       0.42      0.50      0.46      1191
weighted avg       0.70      0.84      0.77      1191

AUC:
0.8994528795811518


In [ ]:
current_run = mlflow.start_run()

#voting (hard)
from sklearn.ensemble import VotingClassifier

clf1 = RandomForestClassifier(n_estimators=100, random_state=42)
clf2 = AdaBoostClassifier(estimator=base_estimator, n_estimators=100, random_state=42)
clf3 = XGBClassifier()

vc_hard = VotingClassifier(estimators=[('rf', clf1), ('ada', clf2), ('xgb', clf3)], voting='hard')
vc_hard.fit(X_train,Y_train)

predictions = vc_hard.predict(X_test)
print(classification_report(Y_test, predictions))

mlflow.end_run()



2023/08/21 15:25:02 WARNING mlflow.utils: Truncated the value of the key `estimators`. Truncated value: `[('rf', RandomForestClassifier(random_state=42)), ('ada', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                   n_estimators=100, random_state=42)), ('xgb', XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_id=None, grow_...`
2023/08/21 15:25:02 WARNING mlflow.utils: Truncated the value of the key `xgb`. Truncated value: `XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_

              precision    recall  f1-score   support

           0       0.90      0.96      0.93      1000
           1       0.66      0.41      0.51       191

    accuracy                           0.87      1191
   macro avg       0.78      0.69      0.72      1191
weighted avg       0.86      0.87      0.86      1191



In [ ]:
current_run = mlflow.start_run()

#voting (soft)
clf1 = RandomForestClassifier(n_estimators=100, random_state=42)
clf2 = AdaBoostClassifier(estimator=base_estimator, n_estimators=100, random_state=42)
clf3 = XGBClassifier()

vc_soft = VotingClassifier(estimators=[('rf', clf1), ('ada', clf2), ('xgb', clf3)], voting='soft')
vc_soft.fit(X_train,Y_train)

predictions = vc_soft.predict(X_test)
print(classification_report(Y_test, predictions))

mlflow.end_run()

2023/08/21 15:25:17 WARNING mlflow.utils: Truncated the value of the key `estimators`. Truncated value: `[('rf', RandomForestClassifier(random_state=42)), ('ada', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                   n_estimators=100, random_state=42)), ('xgb', XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_id=None, grow_...`
2023/08/21 15:25:17 WARNING mlflow.utils: Truncated the value of the key `xgb`. Truncated value: `XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_

              precision    recall  f1-score   support

           0       0.90      0.96      0.93      1000
           1       0.67      0.41      0.51       191

    accuracy                           0.87      1191
   macro avg       0.78      0.69      0.72      1191
weighted avg       0.86      0.87      0.86      1191



In [ ]:
current_run = mlflow.start_run()

#voting classifier (hard) on oversampled data

clf1 = RandomForestClassifier(n_estimators=100, random_state=42)
clf2 = AdaBoostClassifier(estimator=base_estimator, n_estimators=100, random_state=42)
clf3 = XGBClassifier()

vc_hard_ros = VotingClassifier(estimators=[('rf', clf1), ('ada', clf2), ('xgb', clf3)], voting='hard')
vc_hard_ros.fit(X_res, Y_res)

predictions = vc_hard_ros.predict(X_test)
print(classification_report(Y_test, predictions))

auc = roc_auc_score(Y_test, y_pred_proba)
print("AUC:")
print(auc)

mlflow.end_run()

2023/08/21 15:25:31 WARNING mlflow.utils: Truncated the value of the key `estimators`. Truncated value: `[('rf', RandomForestClassifier(random_state=42)), ('ada', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                   n_estimators=100, random_state=42)), ('xgb', XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_id=None, grow_...`
2023/08/21 15:25:31 WARNING mlflow.utils: Truncated the value of the key `xgb`. Truncated value: `XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_

              precision    recall  f1-score   support

           0       0.93      0.91      0.92      1000
           1       0.56      0.62      0.59       191

    accuracy                           0.86      1191
   macro avg       0.75      0.77      0.75      1191
weighted avg       0.87      0.86      0.87      1191



In [ ]:
current_run = mlflow.start_run()

#voting classifier (soft) on oversampled data

clf1 = RandomForestClassifier(n_estimators=100, random_state=42)
clf2 = AdaBoostClassifier(estimator=base_estimator, n_estimators=100, random_state=42)
clf3 = XGBClassifier()

vc_soft_ros = VotingClassifier(estimators=[('rf', clf1), ('ada', clf2), ('xgb', clf3)], voting='soft')
vc_soft_ros.fit(X_res, Y_res)

predictions = vc_soft_ros.predict(X_test)
print(classification_report(Y_test, predictions))

mlflow.end_run()

2023/08/21 15:25:46 WARNING mlflow.utils: Truncated the value of the key `estimators`. Truncated value: `[('rf', RandomForestClassifier(random_state=42)), ('ada', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                   n_estimators=100, random_state=42)), ('xgb', XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_id=None, grow_...`
2023/08/21 15:25:46 WARNING mlflow.utils: Truncated the value of the key `xgb`. Truncated value: `XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_

              precision    recall  f1-score   support

           0       0.92      0.92      0.92      1000
           1       0.58      0.57      0.57       191

    accuracy                           0.86      1191
   macro avg       0.75      0.74      0.75      1191
weighted avg       0.86      0.86      0.86      1191



voting - pushes f1 score to 0.59 for '1' but at the expense of precision

#Training using Feature Set B

In [ ]:
#stratify split using feature set B

#initialization a generator
strat_shuff_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Get the index values from the generator
train_idx, test_idx = next(strat_shuff_split.split(df[set_B], df['hit']))
train_idx, test_idx

X_train_B = df.loc[train_idx, set_B]
Y_train_B = df.loc[train_idx, 'hit']

X_test_B = df.loc[test_idx, set_B]
Y_test_B = df.loc[test_idx, 'hit']

In [ ]:
current_run = mlflow.start_run()

# Initialize the Random Forest classifier
rf_B = RandomForestClassifier(n_estimators=100, random_state=42)

# Perform cross-validation with 5 folds (you can adjust the number of folds)
cv_scores = cross_val_score(rf_B, X_train_B, Y_train_B, cv=5, scoring='accuracy')
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

rf_B.fit(X_train_B,Y_train_B)

predictions = rf_B.predict(X_test_B)
print('Random Forest accuracy', accuracy_score(Y_test_B, predictions))

print("Classification Report:")
print(classification_report(Y_test_B, predictions))


mlflow.end_run()

Cross-validation scores: [0.83840504 0.83315845 0.83840504 0.82352941 0.83088235]
Mean CV score: 0.8328760570335165
Random Forest accuracy 0.8295549958018472
Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.94      0.90      1000
           1       0.45      0.27      0.34       191

    accuracy                           0.83      1191
   macro avg       0.66      0.60      0.62      1191
weighted avg       0.80      0.83      0.81      1191



In [ ]:
current_run = mlflow.start_run()

#Ada Boost
# Initialize the AdaBoost classifier with DecisionTreeClassifier as the base estimator
base_estimator = DecisionTreeClassifier(max_depth=1)
ada_B = AdaBoostClassifier(estimator=base_estimator, n_estimators=100, random_state=42)

# Perform cross-validation with 5 folds (you can adjust the number of folds)
cv_scores = cross_val_score(ada_B, X_train_B, Y_train_B, cv=5, scoring='accuracy')
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

ada_B.fit(X_train_B,Y_train_B)

predictions = ada_B.predict(X_test_B)
print('Ada Boost accuracy', accuracy_score(Y_test_B, predictions))

print("Classification Report:")
print(classification_report(Y_test_B, predictions))

mlflow.end_run()

Cross-validation scores: [0.84260231 0.83525708 0.84679958 0.84663866 0.84033613]
Mean CV score: 0.8423267523168765
Ada Boost accuracy 0.836272040302267
Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.95      0.91      1000
           1       0.48      0.23      0.31       191

    accuracy                           0.84      1191
   macro avg       0.67      0.59      0.61      1191
weighted avg       0.80      0.84      0.81      1191



In [ ]:
current_run = mlflow.start_run()

#XG Boost
xgb_B = XGBClassifier()
cv_scores = cross_val_score(xgb_B, X_train_B, Y_train_B, cv=5, scoring='accuracy')

# Print the cross-validation scores
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())

xgb_B.fit(X_train_B, Y_train_B)

predictions = xgb_B.predict(X_test_B)
print('XG Boost boost accuracy', accuracy_score(Y_test_B, predictions))

print("Classification Report:")
print(classification_report(Y_test_B, predictions))

mlflow.end_run()

Cross-validation scores: [0.83735572 0.8247639  0.83840504 0.84033613 0.82457983]
Mean CV score: 0.8330881250716446
XG Boost boost accuracy 0.8337531486146096
Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.94      0.90      1000
           1       0.47      0.29      0.36       191

    accuracy                           0.83      1191
   macro avg       0.67      0.61      0.63      1191
weighted avg       0.81      0.83      0.82      1191



In [ ]:
#current_run = mlflow.start_run()

# Create a pipeline
pipeline_svc_B = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', SVC(kernel='rbf', C=0.01, gamma=0.01,random_state=42))
])

# Cross-validation
cv_scores = cross_val_score(pipeline_svc_B, X_train_B, Y_train_B, cv=5)

# Train the pipeline on the entire training set
pipeline_svc_B.fit(X_train_B, Y_train_B)

mean_cv_score = np.mean(cv_scores)
std_cv_score = np.std(cv_scores)

print(mean_cv_score)
print(std_cv_score)

predictions = pipeline_svc_B.predict(X_test_B)
print('SVM accuracy', accuracy_score(Y_test_B, predictions))

print("Classification Report:")
print(classification_report(Y_test_B, predictions))

#mlflow.end_run()

2023/08/21 15:26:36 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'eccd0a45308441db92d3a14eade8d3b6', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


0.8398070665831915
0.0004319821074154611
SVM accuracy 0.8396305625524769
Classification Report:
              precision    recall  f1-score   support

           0       0.84      1.00      0.91      1000
           1       0.00      0.00      0.00       191

    accuracy                           0.84      1191
   macro avg       0.42      0.50      0.46      1191
weighted avg       0.70      0.84      0.77      1191



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
